In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

In [ ]:
ratings = pd.read_csv('ratings.csv')
movies = pd.read_csv('movies.csv')
tags = pd.read_csv('tags.csv')

In [ ]:
print(f"Ratings dataset shape: {ratings.shape}")
print(f"Movies dataset shape: {movies.shape}")
print(f"Tags dataset shape: {tags.shape}")

Ratings dataset shape: (100836, 4)
Movies dataset shape: (9742, 3)
Tags dataset shape: (3683, 4)


In [ ]:
print("\nRatings (User-Item Interactions):")
print(ratings.head(10))

print("\nMovies (Metadata):")
print(movies.head(10))



Ratings (User-Item Interactions):
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
5       1       70     3.0  964982400
6       1      101     5.0  964980868
7       1      110     4.0  964982176
8       1      151     5.0  964984041
9       1      157     5.0  964984100

Movies (Metadata):
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   
5        6                         Heat (1995)   
6        7                      Sabrina (1995)   
7        8                 Tom and Huck (1995)   
8        9                 Sudden Death (1995)   
9       10                 

In [ ]:
print("\n--- 3. MERGING TABLES (PIPELINE) ---")
merged_data = pd.merge(ratings, movies, on='movieId', how='inner')
print("Merged Data (Ratings + Movie Titles):")
print(merged_data[['userId', 'movieId', 'rating', 'title', 'genres']].head(5))


--- 3. MERGING TABLES (PIPELINE) ---
Merged Data (Ratings + Movie Titles):
   userId  movieId  rating                        title  \
0       1        1     4.0             Toy Story (1995)   
1       1        3     4.0      Grumpier Old Men (1995)   
2       1        6     4.0                  Heat (1995)   
3       1       47     5.0  Seven (a.k.a. Se7en) (1995)   
4       1       50     5.0   Usual Suspects, The (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                               Comedy|Romance  
2                        Action|Crime|Thriller  
3                             Mystery|Thriller  
4                       Crime|Mystery|Thriller  


In [ ]:
num_users = ratings['userId'].nunique()
num_movies = ratings['movieId'].nunique()
num_ratings = len(ratings)

print(f"Total Users: {num_users}")
print(f"Total Movies Rated: {num_movies}")
print(f"Total Ratings: {num_ratings}")

Total Users: 610
Total Movies Rated: 9724
Total Ratings: 100836


In [ ]:
sparsity = (1 - (num_ratings / (num_users * num_movies))) * 100
print(f"Matrix Sparsity: {sparsity:.2f}% empty")
print(f"-> Meaning we only know {100 - sparsity:.2f}% of what users think of these movies.")

Matrix Sparsity: 98.30% empty
-> Meaning we only know 1.70% of what users think of these movies.


In [ ]:
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error

# 1. Create the matrix and get the global average rating
user_item_matrix = ratings.pivot(index='userId', columns='movieId', values='rating')
global_mean = ratings['rating'].mean()

# 2. Fill NaNs with the global mean (instead of 0)
# This prevents the algorithm from thinking unrated movies are rated 0.
matrix_filled = user_item_matrix.fillna(global_mean).values

# 3. Deploy SVD
svd = TruncatedSVD(n_components=20, random_state=42)
U = svd.fit_transform(matrix_filled)
Sigma = np.diag(svd.singular_values_)
Vt = svd.components_

# 4. Predict ratings
predicted_ratings = np.dot(np.dot(U, Sigma), Vt)

# 5. THE MAGIC FIX: Clip predictions to the valid 1-5 range
predicted_ratings = np.clip(predicted_ratings, 1.0, 5.0)

# 6. Evaluate using RMSE
actual_ratings = user_item_matrix.values
mask = ~np.isnan(actual_ratings) # Only look at real ratings

actual_values = actual_ratings[mask]
predicted_values = predicted_ratings[mask]

rmse = np.sqrt(mean_squared_error(actual_values, predicted_values))

print("==========================================")
print("FINAL EVALUATION FOR EXPERIMENT 10")
print("==========================================")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print("==========================================")

FINAL EVALUATION FOR EXPERIMENT 10
Root Mean Squared Error (RMSE): 1.8254


In [ ]:
# 1. Install the industry-standard library
!pip install scikit-surprise

import pandas as pd
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy

# 2. Load data (Surprise needs a specific format)
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# 3. Split into 80% train, 20% test (This is how real ML is evaluated)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# 4. Deploy SVD (Surprise handles all the NaNs and biases automatically!)
algo = SVD(n_factors=20, random_state=42)

# 5. Train the model
algo.fit(trainset)

# 6. Predict on the hidden 20% test set
predictions = algo.test(testset)

# 7. Evaluate using RMSE
rmse = accuracy.rmse(predictions)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 18.8 MB/s  0:00:00
RMSE: 0.8780
